# Sentiment Analysis: EDA and Analysis

Ten notebook pokazuje podstawową analizę danych użytych w projekcie.

Co tu znajdziesz:
- wczytanie danych z `data/raw`,
- sprawdzenie rozkładu klas,
- analizę długości tekstów,
- kilka przykładów zdań,
- miejsce na późniejsze porównanie wyników LSTM i Transformera.

## 1. Importy i ustawienia

W tej sekcji ładujemy biblioteki i definiujemy ścieżki do plików JSONL.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
RESULTS_DIR = PROJECT_ROOT / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'amazon_reviews_multi_en_train.jsonl'
VAL_PATH = DATA_DIR / 'amazon_reviews_multi_en_validation.jsonl'
TEST_PATH = DATA_DIR / 'amazon_reviews_multi_en_test.jsonl'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('TRAIN_PATH exists:', TRAIN_PATH.exists())
print('VAL_PATH exists:', VAL_PATH.exists())
print('TEST_PATH exists:', TEST_PATH.exists())

## 2. Wczytanie danych

Pliki są w formacie JSONL, czyli każdy wiersz to jeden rekord.

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with path.open('r', encoding='utf-8') as file:
        for line in file:
            records.append(json.loads(line))
    return pd.DataFrame(records)

train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print('train shape:', train_df.shape)
print('validation shape:', val_df.shape)
print('test shape:', test_df.shape)

display(train_df.head())

## 3. Jak wyglądają kolumny

Sprawdzamy podstawowy układ danych i typy wartości.

In [ ]:
print(train_df.columns.tolist())
print(train_df.dtypes)

missing = train_df.isna().sum()
print('Missing values in train:')
print(missing)

## 4. Rozkład klas

To ważne, bo niezbalansowane klasy mogą mocno wpływać na wyniki modelu.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
splits = [('Train', train_df), ('Validation', val_df), ('Test', test_df)]
for ax, (name, df) in zip(axes, splits):
    counts = df['label'].value_counts().sort_index()
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, color='#4C72B0')
    ax.set_title(f'{name} class distribution')
    ax.set_xlabel('label')
    ax.set_ylabel('count')
    for i, value in enumerate(counts.values):
        ax.text(i, value, str(value), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train_df['label'].value_counts(normalize=True).sort_index().rename('ratio')

## 5. Długość tekstów

Sprawdzamy, czy większość recenzji jest krótka czy długa. To pomaga dobrać `MAX_LENGTH`.

In [ ]:
def text_length_stats(df: pd.DataFrame) -> pd.DataFrame:
    lengths = df['text'].fillna('').astype(str).str.split().str.len()
    return lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame(name='length')

length_stats = text_length_stats(train_df)
display(length_stats)

train_lengths = train_df['text'].fillna('').astype(str).str.split().str.len()
val_lengths = val_df['text'].fillna('').astype(str).str.split().str.len()
test_lengths = test_df['text'].fillna('').astype(str).str.split().str.len()

plt.figure(figsize=(10, 5))
sns.histplot(train_lengths, bins=50, kde=True, color='#55A868', label='train', alpha=0.5)
sns.histplot(val_lengths, bins=50, kde=False, color='#C44E52', label='validation', alpha=0.35)
sns.histplot(test_lengths, bins=50, kde=False, color='#8172B2', label='test', alpha=0.25)
plt.axvline(128, color='black', linestyle='--', linewidth=1, label='MAX_LENGTH=128')
plt.title('Text length distribution (number of words)')
plt.xlabel('number of words')
plt.ylabel('count')
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Przykładowe teksty

Tu sprawdzamy, jak wyglądają pojedyncze recenzje dla różnych klas.

In [ ]:
label_names = {0: 'negative', 1: 'somewhat negative', 2: 'neutral', 3: 'somewhat positive', 4: 'positive'}

for label in sorted(train_df['label'].unique()):
    sample = train_df[train_df['label'] == label].sample(2, random_state=42)
    print(f"=== label {label} ({label_names.get(label, 'unknown')}) ===")
    for idx, row in sample.iterrows():
        print('---')
        print(row['text'][:500])
        print()

## 7. Szybka analiza słów

Sprawdzamy najczęstsze słowa po prostym tokenizowaniu. To pomaga zrozumieć, czy teksty mają typowe słownictwo recenzji.

In [ ]:
import re
from collections import Counter

def simple_tokenize(text: str):
    return re.findall(r'\w+', str(text).lower())

counter = Counter()
for text in train_df['text'].head(20000):
    counter.update(simple_tokenize(text))

top_words = pd.DataFrame(counter.most_common(20), columns=['word', 'count'])
display(top_words)

plt.figure(figsize=(12, 5))
sns.barplot(data=top_words, x='word', y='count', color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 words in training data (sampled)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Miejsce na wyniki treningu

W tej sekcji można później wczytać zapisane logi treningowe i porównać LSTM z Transformerem.

Na teraz notebook tylko sprawdza, czy istnieją jakieś pliki wynikowe.

In [ ]:
result_files = list(RESULTS_DIR.glob('*'))
print('Files in results/:')
for path in result_files:
    print('-', path.name)

if not result_files:
    print('Brak jeszcze zapisanych wyników treningu. To miejsce jest gotowe na późniejsze logi i wykresy.')

## 9. Wnioski

Na podstawie tego notebooka można sprawdzić:
- czy klasy są zbalansowane,
- jak długie są teksty,
- jakie słowa dominują w danych,
- czy `MAX_LENGTH=128` ma sens,
- gdzie później wstawić porównanie modeli i wykresy z treningu.